In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("debeshjha1/kvasirseg")

print("Path to dataset files:", path)

100%|██████████| 144M/144M [00:01<00:00, 100MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3


In [ ]:
!pip install -q timm albumentations

In [ ]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split

from tqdm.auto import tqdm

In [ ]:
SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 20
LEARNING_RATE = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
print("Dataset path:")
print(path)

print("\nFolders inside dataset:")

for root, dirs, files in os.walk(path):
    level = root.replace(path, "").count(os.sep)

    if level <= 2:
        print(
            "  " * level + os.path.basename(root) + "/"
        )

Dataset path:
/root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3

Folders inside dataset:
3/
  Kvasir-SEG/
    Kvasir-SEG/


In [ ]:
image_candidates = []
mask_candidates = []

for root, dirs, files in os.walk(path):

    folder_name = os.path.basename(root).lower()

    if folder_name == "images":
        image_candidates.append(root)

    if folder_name == "masks":
        mask_candidates.append(root)

print("Image folders found:", image_candidates)
print("Mask folders found:", mask_candidates)

Image folders found: ['/root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3/Kvasir-SEG/Kvasir-SEG/images']
Mask folders found: ['/root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3/Kvasir-SEG/Kvasir-SEG/masks']


In [ ]:
IMAGE_DIR = image_candidates[0]
MASK_DIR = mask_candidates[0]

print("\nIMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)


IMAGE_DIR: /root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3/Kvasir-SEG/Kvasir-SEG/images
MASK_DIR: /root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3/Kvasir-SEG/Kvasir-SEG/masks


In [ ]:
image_paths = sorted(
    glob.glob(os.path.join(IMAGE_DIR, "*.jpg")) +
    glob.glob(os.path.join(IMAGE_DIR, "*.jpeg")) +
    glob.glob(os.path.join(IMAGE_DIR, "*.png"))
)

mask_paths = sorted(
    glob.glob(os.path.join(MASK_DIR, "*.png")) +
    glob.glob(os.path.join(MASK_DIR, "*.jpg")) +
    glob.glob(os.path.join(MASK_DIR, "*.jpeg"))
)

print("Number of images:", len(image_paths))
print("Number of masks:", len(mask_paths))

Number of images: 1000
Number of masks: 1000


In [ ]:
image_dict = {
    os.path.splitext(
        os.path.basename(p)
    )[0]: p
    for p in image_paths
}

mask_dict = {
    os.path.splitext(
        os.path.basename(p)
    )[0]: p
    for p in mask_paths
}

common_names = sorted(
    set(image_dict.keys()) &
    set(mask_dict.keys())
)

images = [
    image_dict[name]
    for name in common_names
]

masks = [
    mask_dict[name]
    for name in common_names
]

print("Matched image-mask pairs:", len(images))

Matched image-mask pairs: 1000


In [ ]:
train_images, temp_images, train_masks, temp_masks = train_test_split(
    images,
    masks,
    test_size=0.20,
    random_state=SEED
)

val_images, test_images, val_masks, test_masks = train_test_split(
    temp_images,
    temp_masks,
    test_size=0.50,
    random_state=SEED
)

print("Training:", len(train_images))
print("Validation:", len(val_images))
print("Testing:", len(test_images))

Training: 800
Validation: 100
Testing: 100


In [ ]:
train_transform = A.Compose([

    A.Resize(
        IMG_SIZE,
        IMG_SIZE
    ),

    A.HorizontalFlip(
        p=0.5
    ),

    A.VerticalFlip(
        p=0.2
    ),

    A.RandomRotate90(
        p=0.5
    ),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=15,
        p=0.5
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


val_transform = A.Compose([

    A.Resize(
        IMG_SIZE,
        IMG_SIZE
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
class KvasirDataset(Dataset):

    def __init__(
        self,
        image_paths,
        mask_paths,
        transform=None
    ):

        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform


    def __len__(self):

        return len(self.image_paths)


    def __getitem__(self, idx):

        # Load image
        image = np.array(
            Image.open(
                self.image_paths[idx]
            ).convert("RGB")
        )

        # Load mask
        mask = np.array(
            Image.open(
                self.mask_paths[idx]
            ).convert("L")
        )

        # Binary mask
        mask = (
            mask > 127
        ).astype(np.uint8)

        # Apply transformations
        if self.transform:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        mask = mask.float()

        # Add channel dimension
        if mask.ndim == 2:

            mask = mask.unsqueeze(0)

        return image, mask

In [ ]:
train_dataset = KvasirDataset(
    train_images,
    train_masks,
    train_transform
)

val_dataset = KvasirDataset(
    val_images,
    val_masks,
    val_transform
)

test_dataset = KvasirDataset(
    test_images,
    test_masks,
    val_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 200
Validation batches: 25
Test batches: 25


In [ ]:
class AttentionBlock(nn.Module):

    def __init__(
        self,
        channels,
        num_heads=8
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(
            channels
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(
            channels
        )

        self.mlp = nn.Sequential(

            nn.Linear(
                channels,
                channels * 4
            ),

            nn.GELU(),

            nn.Linear(
                channels * 4,
                channels
            )
        )


    def forward(self, x):

        B, C, H, W = x.shape

        x_flat = (
            x.flatten(2)
             .transpose(1, 2)
        )

        residual = x_flat

        x_norm = self.norm1(
            x_flat
        )

        attention_output, _ = self.attention(
            x_norm,
            x_norm,
            x_norm
        )

        x_flat = (
            residual +
            attention_output
        )

        x_flat = (
            x_flat +
            self.mlp(
                self.norm2(x_flat)
            )
        )

        x = (
            x_flat
            .transpose(1, 2)
            .reshape(B, C, H, W)
        )

        return x

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels + skip_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(
            out_channels
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(
            out_channels
        )


    def forward(
        self,
        x,
        skip
    ):

        x = F.interpolate(
            x,
            size=skip.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = F.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = F.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        return x

In [ ]:
class TransResNet(nn.Module):

    def __init__(self):

        super().__init__()


        # ==========================================
        # RESNET BRANCH
        # ==========================================

        self.resnet = timm.create_model(
            "resnet18",
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4)
        )


        # ==========================================
        # SWIN TRANSFORMER BRANCH
        # ==========================================

        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3)
        )


        # ==========================================
        # FEATURE FUSION
        # ==========================================

        self.fuse1 = nn.Conv2d(
            64 + 96,
            64,
            kernel_size=1
        )

        self.fuse2 = nn.Conv2d(
            128 + 192,
            128,
            kernel_size=1
        )

        self.fuse3 = nn.Conv2d(
            256 + 384,
            256,
            kernel_size=1
        )

        self.fuse4 = nn.Conv2d(
            512 + 768,
            512,
            kernel_size=1
        )


        # ==========================================
        # ATTENTION
        # ==========================================

        self.attention = AttentionBlock(
            channels=512,
            num_heads=8
        )


        # ==========================================
        # DECODER
        # ==========================================

        self.decoder4 = DecoderBlock(
            512,
            256,
            256
        )

        self.decoder3 = DecoderBlock(
            256,
            128,
            128
        )

        self.decoder2 = DecoderBlock(
            128,
            64,
            64
        )

        self.decoder1 = DecoderBlock(
            64,
            64,
            32
        )


        # ==========================================
        # OUTPUT
        # ==========================================

        self.final_layer = nn.Sequential(

            nn.Conv2d(
                32,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(16),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                16,
                1,
                kernel_size=1
            )
        )


    def forward(self, x):

        # ==========================================
        # RESNET FEATURES
        # ==========================================

        r = self.resnet(x)

        r1 = r[1]
        r2 = r[2]
        r3 = r[3]
        r4 = r[4]


        # ==========================================
        # SWIN FEATURES
        # ==========================================

        s = self.swin(x)

        s1 = s[0]
        s2 = s[1]
        s3 = s[2]
        s4 = s[3]


        # Swin: B,H,W,C
        # Convert: B,C,H,W

        s1 = s1.permute(
            0, 3, 1, 2
        )

        s2 = s2.permute(
            0, 3, 1, 2
        )

        s3 = s3.permute(
            0, 3, 1, 2
        )

        s4 = s4.permute(
            0, 3, 1, 2
        )


        # ==========================================
        # RESIZE SWIN FEATURES
        # ==========================================

        s1 = F.interpolate(
            s1,
            size=r1.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        s2 = F.interpolate(
            s2,
            size=r2.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        s3 = F.interpolate(
            s3,
            size=r3.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        s4 = F.interpolate(
            s4,
            size=r4.shape[2:],
            mode="bilinear",
            align_corners=False
        )


        # ==========================================
        # FEATURE GRAFTING / FUSION
        # ==========================================

        f1 = self.fuse1(
            torch.cat(
                [r1, s1],
                dim=1
            )
        )

        f2 = self.fuse2(
            torch.cat(
                [r2, s2],
                dim=1
            )
        )

        f3 = self.fuse3(
            torch.cat(
                [r3, s3],
                dim=1
            )
        )

        f4 = self.fuse4(
            torch.cat(
                [r4, s4],
                dim=1
            )
        )


        # ==========================================
        # GLOBAL ATTENTION
        # ==========================================

        f4 = self.attention(f4)


        # ==========================================
        # DECODER
        # ==========================================

        d4 = self.decoder4(
            f4,
            f3
        )

        d3 = self.decoder3(
            d4,
            f2
        )

        d2 = self.decoder2(
            d3,
            f1
        )


        # ==========================================
        # FINAL UPSAMPLING
        # ==========================================

        d2 = F.interpolate(
            d2,
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )

        # Use f1 as skip at matching resolution
        f1_up = F.interpolate(
            f1,
            size=d2.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        d1 = self.decoder1(
            d2,
            f1_up
        )


        # ==========================================
        # OUTPUT 256x256
        # ==========================================

        output = F.interpolate(
            d1,
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear",
            align_corners=False
        )

        output = self.final_layer(
            output
        )

        return output

In [ ]:
model = TransResNet().to(device)

print("TransResNet created successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

TransResNet created successfully.


In [ ]:
class DiceLoss(nn.Module):

    def __init__(
        self,
        smooth=1e-6
    ):

        super().__init__()

        self.smooth = smooth


    def forward(
        self,
        logits,
        targets
    ):

        probs = torch.sigmoid(
            logits
        )

        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (
            probs * targets
        ).sum()

        dice = (
            2 * intersection +
            self.smooth
        ) / (
            probs.sum() +
            targets.sum() +
            self.smooth
        )

        return 1 - dice


dice_loss = DiceLoss()

bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(
    prediction,
    target
):

    return (
        bce_loss(
            prediction,
            target
        )
        +
        dice_loss(
            prediction,
            target
        )
    )

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

In [ ]:
def validate(
    model,
    loader
):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = combined_loss(
                outputs,
                masks
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, masks in progress:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = combined_loss(
            outputs,
            masks
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        running_loss /
        len(train_loader)
    )

    val_loss = validate(
        model,
        val_loader
    )

    print(
        f"\nEpoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Validation Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "transresnet_best.pth"
        )

        print("Best model saved.")

Epoch 1/20:   0%|          | 0/200 [00:00<?, ?it/s]


Epoch 1/20
Train Loss: 1.0261
Validation Loss: 0.9075
Best model saved.


Epoch 2/20:   0%|          | 0/200 [00:00<?, ?it/s]


Epoch 2/20
Train Loss: 0.9066
Validation Loss: 0.8123
Best model saved.


Epoch 3/20:   0%|          | 0/200 [00:00<?, ?it/s]


Epoch 3/20
Train Loss: 0.8156
Validation Loss: 0.8595


... (36 outputs left out)


Epoch 19/20
Train Loss: 0.2585
Validation Loss: 0.3128


Epoch 20/20:   0%|          | 0/200 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d9e5b5ca7a0><function _MultiProcessingDataLoaderIter.__del__ at 0x7d9e5b5ca7a0>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
if w.is_alive():    if w.is_alive():

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only te


Epoch 20/20
Train Loss: 0.2564
Validation Loss: 0.2531
Best model saved.


In [ ]:
model.load_state_dict(
    torch.load(
        "transresnet_best.pth",
        map_location=device
    )
)

model.eval()

print("Best TransResNet model loaded.")

Best TransResNet model loaded.


In [ ]:
def calculate_metrics(
    model,
    loader
):

    model.eval()

    total_tp = 0
    total_tn = 0
    total_fp = 0
    total_fn = 0

    with torch.no_grad():

        for images, masks in tqdm(
            loader,
            desc="Testing"
        ):

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            probabilities = torch.sigmoid(
                outputs
            )

            predictions = (
                probabilities >= 0.5
            ).float()

            predictions = predictions.view(-1)
            masks = masks.view(-1)

            tp = (
                (predictions == 1) &
                (masks == 1)
            ).sum().item()

            tn = (
                (predictions == 0) &
                (masks == 0)
            ).sum().item()

            fp = (
                (predictions == 1) &
                (masks == 0)
            ).sum().item()

            fn = (
                (predictions == 0) &
                (masks == 1)
            ).sum().item()

            total_tp += tp
            total_tn += tn
            total_fp += fp
            total_fn += fn


    # DSC
    dsc = (
        2 * total_tp
    ) / (
        2 * total_tp +
        total_fp +
        total_fn +
        1e-8
    )


    # IoU
    iou = (
        total_tp
    ) / (
        total_tp +
        total_fp +
        total_fn +
        1e-8
    )


    # Precision
    precision = (
        total_tp
    ) / (
        total_tp +
        total_fp +
        1e-8
    )


    # Recall
    recall = (
        total_tp
    ) / (
        total_tp +
        total_fn +
        1e-8
    )


    # F1
    f1 = (
        2 * precision * recall
    ) / (
        precision +
        recall +
        1e-8
    )


    # Accuracy
    accuracy = (
        total_tp +
        total_tn
    ) / (
        total_tp +
        total_tn +
        total_fp +
        total_fn +
        1e-8
    )


    return (
        dsc,
        iou,
        f1,
        accuracy
    )

In [ ]:
dsc, iou, f1, accuracy = calculate_metrics(
    model,
    test_loader
)

print("\n")
print("TransResNet Test Results")
print("=========================")

print(
    f"Dice Similarity Coefficient (DSC): {dsc:.4f}"
)

print(
    f"Intersection over Union (IoU):     {iou:.4f}"
)

print(
    f"F1 Score:                          {f1:.4f}"
)

print(
    f"Accuracy:                          {accuracy:.4f}"
)

Testing:   0%|          | 0/25 [00:00<?, ?it/s]



TransResNet Test Results
Dice Similarity Coefficient (DSC): 0.8903
Intersection over Union (IoU):     0.8023
F1 Score:                          0.8903
Accuracy:                          0.9627
